# Challenge 1 - Tic Tac Toe

In this lab you will perform deep learning analysis on a dataset of playing [Tic Tac Toe](https://en.wikipedia.org/wiki/Tic-tac-toe).

There are 9 grids in Tic Tac Toe that are coded as the following picture shows:

![Tic Tac Toe Grids](tttboard.jpg)

In the first 9 columns of the dataset you can find which marks (`x` or `o`) exist in the grids. If there is no mark in a certain grid, it is labeled as `b`. The last column is `class` which tells you whether Player X (who always moves first in Tic Tac Toe) wins in this configuration. Note that when `class` has the value `False`, it means either Player O wins the game or it ends up as a draw.

In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder

Follow the steps suggested below to conduct a neural network analysis using Tensorflow and Keras. You will build a deep learning model to predict whether Player X wins the game or not.

## Step 1: Data Engineering

This dataset is almost in the ready-to-use state so you do not need to worry about missing values and so on. Still, some simple data engineering is needed.

1. Read `tic-tac-toe.csv` into a dataframe.
1. Inspect the dataset. Determine if the dataset is reliable by eyeballing the data.
1. Convert the categorical values to numeric in all columns.
1. Separate the inputs and output.
1. Normalize the input data.

In [2]:
tttoe_df = pd.read_csv('tic-tac-toe.csv')
tttoe_df.head()

,TL,TM,TR,ML,MM,MR,BL,BM,BR,class
0,x,x,x,x,o,o,x,o,o,True
1,x,x,x,x,o,o,o,x,o,True
2,x,x,x,x,o,o,o,o,x,True
3,x,x,x,x,o,o,o,b,b,True
4,x,x,x,x,o,o,b,o,b,True


In [3]:
print(pd.Series(tttoe_df.drop(columns='class').values.flatten()).unique()) # ['x' 'o' 'b']
print(tttoe_df.describe(), # 958 entries and 10 cols (9 loc [3x3], 1 result [bool])
      tttoe_df.info()) # 3 unique values in loc cols
print(tttoe_df.isnull().sum()) # No n/a values

le = LabelEncoder()

['x' 'o' 'b']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 958 entries, 0 to 957
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   TL      958 non-null    object
 1   TM      958 non-null    object
 2   TR      958 non-null    object
 3   ML      958 non-null    object
 4   MM      958 non-null    object
 5   MR      958 non-null    object
 6   BL      958 non-null    object
 7   BM      958 non-null    object
 8   BR      958 non-null    object
 9   class   958 non-null    bool  
dtypes: bool(1), object(9)
memory usage: 68.4+ KB
         TL   TM   TR   ML   MM   MR   BL   BM   BR class
count   958  958  958  958  958  958  958  958  958   958
unique    3    3    3    3    3    3    3    3    3     2
top       x    x    x    x    x    x    x    x    x  True
freq    418  378  418  378  458  378  418  378  418   626 None
TL       0
TM       0
TR       0
ML       0
MM       0
MR       0
BL       0
BM       0
BR       0
class

## Step 2: Build Neural Network

To build the neural network, you can refer to your own codes you wrote while following the [Deep Learning with Python, TensorFlow, and Keras tutorial](https://www.youtube.com/watch?v=wQ8BIBpya2k) in the lesson. It's pretty similar to what you will be doing in this lab.

1. Split the training and test data.
1. Create a `Sequential` model.
1. Add several layers to your model. Make sure you use ReLU as the activation function for the middle layers. Use Softmax for the output layer because each output has a single lable and all the label probabilities add up to 1.
1. Compile the model using `adam` as the optimizer and `sparse_categorical_crossentropy` as the loss function. For metrics, use `accuracy` for now.
1. Fit the training data.
1. Evaluate your neural network model with the test data.
1. Save your model as `tic-tac-toe.model`.

In [4]:
from sklearn.model_selection import train_test_split
seed = 49

X, y = tttoe_df.drop(columns='class'), tttoe_df['class']

# 1. Encode categorical columns (x→2, o→1, b→0)
X = tttoe_df.drop(columns='class').copy()
for col in X.columns:
    X[col] = le.fit_transform(X[col])
# 2. Encode target (True→1, False→0)
y = tttoe_df['class'].astype(int)   
# 3. Normalize inputs (values are 0/1/2, so divide by 2)
X = X / 2.0

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=seed
)

n_cols = X.shape[1]
print(X.head())
y.head()

    TL   TM   TR   ML   MM   MR   BL   BM   BR
0  1.0  1.0  1.0  1.0  0.5  0.5  1.0  0.5  0.5
1  1.0  1.0  1.0  1.0  0.5  0.5  0.5  1.0  0.5
2  1.0  1.0  1.0  1.0  0.5  0.5  0.5  0.5  1.0
3  1.0  1.0  1.0  1.0  0.5  0.5  0.5  0.0  0.0
4  1.0  1.0  1.0  1.0  0.5  0.5  0.0  0.5  0.0


0    1
1    1
2    1
3    1
4    1
Name: class, dtype: int32

In [5]:
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense, Dropout

# create model
kr_model = Sequential([
    Dense(64, activation='relu', input_shape=(n_cols,)),
    Dense(64, activation='relu'),
    Dense(2, activation='softmax')
])
# compile
kr_model.compile(optimizer='adam', 
                 loss='sparse_categorical_crossentropy',
                 metrics=['accuracy'])

# Fit
kr_model.fit(X_train, y_train, epochs=50, validation_split=0.1)

# Save
kr_model.save('tic-tac-toe.keras')

Epoch 1/50


c:\Users\paudu\anaconda3\envs\Main\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.6604 - loss: 0.6433 - val_accuracy: 0.5974 - val_loss: 0.6781
Epoch 2/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6647 - loss: 0.6197 - val_accuracy: 0.5974 - val_loss: 0.6501
Epoch 3/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6734 - loss: 0.6046 - val_accuracy: 0.5974 - val_loss: 0.6436
Epoch 4/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6792 - loss: 0.5871 - val_accuracy: 0.6364 - val_loss: 0.6258
Epoch 5/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6996 - loss: 0.5730 - val_accuracy: 0.6364 - val_loss: 0.6235
Epoch 6/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7126 - loss: 0.5571 - val_accuracy: 0.6623 - val_loss: 0.6091
Epoch 7/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7242 - loss: 0.5449 - val_accuracy: 0.6883 - val_loss: 0.5886
Epoch 8/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7431 - loss: 0.5261 - val_accuracy: 0.6883 - val_loss: 0.5802
Ep

## Step 3: Make Predictions

Now load your saved model and use it to make predictions on a few random rows in the test dataset. Check if the predictions are correct.

In [6]:
# Evaluate
kr_model.evaluate(X_test, y_test)

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8333 - loss: 0.3836 


[0.3835698068141937, 0.8333333134651184]

## Step 4: Improve Your Model

Did your model achieve low loss (<0.1) and high accuracy (>0.95)? If not, try to improve your model.

But how? There are so many things you can play with in Tensorflow and in the next challenge you'll learn about these things. But in this challenge, let's just do a few things to see if they will help.

* Add more layers to your model. If the data are complex you need more layers. But don't use more layers than you need. If adding more layers does not improve the model performance you don't need additional layers.
* Adjust the learning rate when you compile the model. This means you will create a custom `tf.keras.optimizers.Adam` instance where you specify the learning rate you want. Then pass the instance to `model.compile` as the optimizer.
    * `tf.keras.optimizers.Adam` [reference](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam).
    * Don't worry if you don't understand what the learning rate does. You'll learn about it in the next challenge.
* Adjust the number of epochs when you fit the training data to the model. Your model performance continues to improve as you train more epochs. But eventually it will reach the ceiling and the performance will stay the same.

In [7]:
from keras.callbacks import EarlyStopping

earlystopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

In [8]:
# create model
kr_model = Sequential([
    Dense(64, activation='relu', input_shape=(n_cols,)),
    Dense(64, activation='relu'),
    Dense(2, activation='softmax')
])

# compile
kr_model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01),
                 loss='sparse_categorical_crossentropy',
                 metrics=['accuracy'])

# Fit
kr_model.fit(X_train, y_train, 
             epochs=100, 
             validation_split=0.1,
             callbacks=earlystopping,
)

# Save
kr_model.save('tic-tac-toe2.keras')

# Evaluate
kr_model.evaluate(X_test, y_test)

Epoch 1/100


c:\Users\paudu\anaconda3\envs\Main\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.6546 - loss: 0.6261 - val_accuracy: 0.6494 - val_loss: 0.6152
Epoch 2/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7300 - loss: 0.5666 - val_accuracy: 0.6494 - val_loss: 0.6019
Epoch 3/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7300 - loss: 0.5358 - val_accuracy: 0.6753 - val_loss: 0.5680
Epoch 4/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7721 - loss: 0.4979 - val_accuracy: 0.7143 - val_loss: 0.5190
Epoch 5/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7983 - loss: 0.4388 - val_accuracy: 0.7273 - val_loss: 0.5332
Epoch 6/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8200 - loss: 0.3937 - val_accuracy: 0.7532 - val_loss: 0.6249
Epoch 7/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7823 - loss: 0.4553 - val_accuracy: 0.7273 - val_loss: 0.5020
Epoch 8/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8331 - loss: 0.3921 - val_accuracy: 0.7792 - val_loss: 0.4

[0.2807510495185852, 0.9166666865348816]

**Which approach(es) did you find helpful to improve your model performance?**

In [9]:
'''
1. Adjusting learning rate (0.01) helped speed up convergence, but also caused overfitting
2. Early stopping with patience of 10 epochs helped mitigate overfitting by restoring best weights
3. Model still did not achieve perfect accuracy on test set, likely due to small dataset size and model capacity
4. Changing epochs to 100 did not improve performance, as early stopping halted training at optimal point 
5. Overall, model achieved good accuracy (~0.90) on test set, but not perfect 
'''

'\n1. Adjusting learning rate (0.01) helped speed up convergence, but also caused overfitting\n2. Early stopping with patience of 10 epochs helped mitigate overfitting by restoring best weights\n3. Model still did not achieve perfect accuracy on test set, likely due to small dataset size and model capacity\n4. Changing epochs to 100 did not improve performance, as early stopping halted training at optimal point \n5. Overall, model achieved good accuracy (~0.90) on test set, but not perfect \n'